# Phase 2 — Checkpointed U-Net and paper-inspired R2AU-Net baselines

Run on the connected Kaggle GPU kernel, in order. This notebook embeds the source
and frozen pilot manifest. Initial runs use seed 42 and 16 base channels; they are
pilot baselines, not exact paper reproductions. Only one GPU is used deliberately.
No training results are claimed until execution and export are verified.

Recovery tests must pass before long training. Checkpoints are local to the live
session until you download/save outputs. Keep the session running until backup
is verified. No snapshots are automatically deleted.


In [2]:
import base64, hashlib, io, json, os, sys, zipfile
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='389b9dc97e3284955155955fd25274d5affcb5efe59bbe3b1837b0a8a1dbde12'
project=Path('/kaggle/working/road_crack_phase2_v1')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        path=(project/name).resolve()
        assert path.is_relative_to(project.resolve())
        if path.exists(): assert path.read_bytes()==z.read(name), 'Code differs: preserve this run and use a new project directory'
    z.extractall(project)
os.chdir(project)
manifest=project/'data/manifests/pilot_v1.json'
assert hashlib.sha256(manifest.read_bytes()).hexdigest()==manifest.with_suffix('.json.sha256').read_text().split()[0]
import subprocess
restoration_report=project/'artifacts/dataset_restore.json'
subprocess.run([sys.executable,'scripts/restore_datasets.py',
    '--manifest',str(manifest),'--sources','configs/data_sources.json',
    '--destination','/kaggle/working/road_crack_phase1/data/raw',
    '--report',str(restoration_report)],check=True)
data_root=Path(json.loads(restoration_report.read_text())['data_root'])
import torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU'
print('GPU:',torch.cuda.get_device_name(0),'Devices:',torch.cuda.device_count(),'Torch:',torch.__version__)
print('Frozen splits:',json.loads(manifest.read_text())['counts'])


Previous data unavailable/incomplete. Restoring pinned repositories to: /kaggle/working/road_crack_phase1/data/raw


HEAD is now at 8202a60 Update README.md
HEAD is now at bd8fbd7 Add files via upload


Restoration complete: verified 657 pairs against the unchanged frozen manifest
GPU: Tesla T4 Devices: 2 Torch: 2.11.0+cu128
Frozen splits: {'source_holdout': 237, 'source_train': 240, 'source_validation': 60, 'target_acquisition': 25, 'target_query': 95}


In [3]:
import subprocess, xml.etree.ElementTree as ET
evidence=project/'artifacts/phase2'
evidence.mkdir(parents=True,exist_ok=True)
test_run=subprocess.run([sys.executable,'-m','pytest','-q','-rA',
    '--junitxml='+str(evidence/'tests.xml')],text=True,capture_output=True)
(evidence/'tests.txt').write_text(test_run.stdout+test_run.stderr)
print(test_run.stdout,test_run.stderr)
assert test_run.returncode==0, 'Recovery tests failed; do not start training'
cases=ET.parse(evidence/'tests.xml').findall('.//testcase')
resume_cases=[c for c in cases if 'test_actual_trainer_mid_epoch_resume' in c.attrib.get('name','')]
assert len(resume_cases)==4 and all(c.find('skipped') is None for c in resume_cases), 'Both CPU and CUDA recovery tests must execute'
(evidence/'requirements.kaggle.lock.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
subprocess.run([sys.executable,'scripts/environment.py'],check=True)
print('Actual U-Net and R2AU trainer recovery tests passed on CPU and CUDA.')


..................                                                       [100%]
==================================== PASSES ====================================
________________ test_actual_trainer_mid_epoch_resume[cpu-unet] ________________
----------------------------- Captured stdout call -----------------------------
{"epoch": 1, "step": 2, "train_loss": 0.8914123177528381, "validation": {"loss": 0.8913527727127075, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "n": 2}, "learning_rate": 0.001}
{"epoch": 2, "step": 4, "train_loss": 0.8913277685642242, "validation": {"loss": 0.8912923336029053, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "n": 2}, "learning_rate": 0.001}
{"epoch": 1, "step": 2, "train_loss": 0.8914123177528381, "validation": {"loss": 0.8913527727127075, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "n": 2}, "learning_rate": 0.001}
{"epoch": 2, "step": 4, "train_loss": 0.8913277685642242, "validation": {"loss": 0.8912923336029053, 

## Training

The following cell starts both pilot baselines (up to 100 epochs each, early
stopping patience 10). Training is FP32, deterministic, batch eight, 256 × 256.
Checkpoints are saved at each epoch and every 30 steps. Rerunning resumes an
existing checkpoint; configurations and dataset hashes must match exactly.

Interrupt requests handled by the training process stop at a complete optimizer
step. A forced kernel kill can lose progress since the latest saved checkpoint.
To stop from another cell/terminal, send SIGTERM to the printed worker PID; avoid
shutting down the kernel/session. If disk space runs low, export then resume on
adequate persistent storage; older snapshots are preserved.


In [4]:
import shutil
assert shutil.disk_usage('/kaggle/working').free>8*1024**3, 'Need at least 8 GiB free before the two pilot runs'
def execute_logged(arguments,log_path):
    with open(log_path,'a',buffering=1) as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        print('Training worker PID:',process.pid,flush=True)
        try:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line)
            code=process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGTERM)
            process.wait(timeout=120)
            raise
    if code: raise RuntimeError('Worker failed; inspect '+str(log_path))
for architecture in ['unet','r2au']:
    run=project/'artifacts/phase2'/architecture
    command=[sys.executable,'scripts/train_baseline.py','--config','configs/baseline_'+architecture+'.json',
             '--manifest',str(manifest),'--data-root',str(data_root),'--run-dir',str(run)]
    if (run/'checkpoints/latest.json').exists(): command.append('--resume')
    execute_logged(command,evidence/(architecture+'_console.log'))
    status=json.loads((run/'status.json').read_text())
    if status['state']!='training_complete':
        print('Stopped safely; export checkpoints before resuming.'); break
    print('Completed',architecture,json.loads((run/'evaluation.json').read_text()))


Training worker PID: 175
{"epoch": 1, "step": 30, "train_loss": 0.9349958896636963, "validation": {"loss": 0.9080382068951924, "dice": 0.2657303669839166, "iou": 0.16494761907185118, "precision": 0.20146030867472292, "recall": 0.5442929408454802, "n": 60}, "learning_rate": 0.001}
{"epoch": 2, "step": 60, "train_loss": 0.973316615819931, "validation": {"loss": 1.0, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "n": 60}, "learning_rate": 0.001}
{"epoch": 3, "step": 90, "train_loss": 1.0, "validation": {"loss": 1.0, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "n": 60}, "learning_rate": 0.001}
{"epoch": 4, "step": 120, "train_loss": 1.0, "validation": {"loss": 1.0, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "n": 60}, "learning_rate": 0.001}
{"epoch": 5, "step": 150, "train_loss": 1.0, "validation": {"loss": 1.0, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "n": 60}, "learning_rate": 0.001}
{"epoch": 6, "step": 180, "train_loss": 1.0, "v

In [5]:
# Profile each completed validation-best model on the same GPU and CPU.
for architecture in ['unet','r2au']:
    run=evidence/architecture
    if not (run/'evaluation.json').exists(): continue
    for device in ['cuda','cpu']:
        execute_logged([sys.executable,'scripts/profile_baseline.py','--run-dir',str(run),
                        '--device',device],evidence/(architecture+'_profile.log'))


Training worker PID: 193
{
  "architecture": "unet",
  "device": "cuda",
  "hardware": "Tesla T4",
  "threads": 4,
  "batch_size": 1,
  "image_size": 256,
  "parameters": 1962625,
  "p50_ms": 2.4852349999946455,
  "p95_ms": 4.344439000078637,
  "warmups": 20,
  "timed_iterations": 200,
  "scope": "model pass only; no image decoding/transfer",
  "recovery_checkpoint_bytes": 23613769,
  "fp32_weight_bytes": 7850500,
  "cuda_peak_allocated_bytes": 46171648,
  "macs": "not yet profiled; do not infer from parameter count"
}
Training worker PID: 202
{
  "architecture": "unet",
  "device": "cpu",
  "hardware": "x86_64",
  "threads": 4,
  "batch_size": 1,
  "image_size": 256,
  "parameters": 1962625,
  "p50_ms": 55.87786149999374,
  "p95_ms": 63.07463899997856,
  "warmups": 20,
  "timed_iterations": 200,
  "scope": "model pass only; no image decoding/transfer",
  "recovery_checkpoint_bytes": 23613769,
  "fp32_weight_bytes": 7850500,
  "cuda_peak_allocated_bytes": null,
  "macs": "not yet profi

## Export before ending the session

Run this cell even after an earlier failure. It exports code, manifest, tests,
logs, evaluation, and the latest/best checkpoint snapshots for each model. Other
historical snapshots remain on Kaggle without deletion. Download the ZIP and
checksum from Kaggle Files/Output, and retain a persistent Kaggle output version.
This cell does not itself verify that an external backup was downloaded.


In [6]:
import zipfile,hashlib
backup=Path('/kaggle/working/phase2_results.zip')
selected=set()
for path in project.rglob('*'):
    if not path.is_file() or '__pycache__' in path.parts or '.pytest_cache' in path.parts: continue
    if path.suffix=='.pt': continue
    selected.add(path)
for architecture in ['unet','r2au']:
    folder=evidence/architecture/'checkpoints'
    for which in ['latest','best']:
        pointer=folder/(which+'.json')
        if pointer.exists():
            entry=json.loads(pointer.read_text())
            path=folder/entry['file']
            assert path.resolve().parent==folder.resolve()
            assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256']
            selected.add(path)
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(selected): z.write(path,path.relative_to(project).as_posix())
digest=hashlib.sha256(backup.read_bytes()).hexdigest()
backup.with_suffix('.zip.sha256').write_text(digest+'  '+backup.name+'\n')
print('Download:',backup)
print('SHA256:',digest)
print('Historical snapshots remain on Kaggle. Latest and best are included in this ZIP.')


Download: /kaggle/working/phase2_results.zip
SHA256: e216010ff251fb773c29a33cc59648d31976610622bf43055b6a92a24c1b2d73
Historical snapshots remain on Kaggle. Latest and best are included in this ZIP.
